In [0]:
%pip install pyyaml

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

CODE_ROOT = "../config"

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

from utils.pipeline_config import (
    get_config,
    load_contract,
    project_contract,
    METADATA,
)

config = get_config("dev")
spark.conf.set("spark.sql.session.timeZone", "UTC")

contract = load_contract(
    spark,
    f"{CODE_ROOT}/source_contracts.yaml",
)

# True only when intentionally resetting this practice setup.
RESET_DEV_TABLES = True

In [0]:
if config.environment != "dev":
    raise RuntimeError(
        "This setup/reset notebook is restricted to dev"
    )

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {config.namespace}"
)

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"{config.namespace}.{config.volume}"
)

tables = [
    config.monitoring_table,
    config.quarantine_table,
    *[
        config.bronze_table(entity)
        for entity in contract["entities"]
    ],
]

if RESET_DEV_TABLES:
    for table in tables:
        spark.sql(f"DROP TABLE IF EXISTS {table}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {config.monitoring_table} (
    environment STRING,
    batch_id STRING,
    entity STRING,
    ingestion_run_id STRING,
    trigger_type STRING,
    load_type STRING,
    status STRING,

    source_uri STRING,
    source_file STRING,
    file_timestamp TIMESTAMP,
    file_checksum STRING,
    target_table STRING,

    contract_version STRING,
    contract_checksum STRING,

    expected_count BIGINT,
    input_count BIGINT,
    accepted_count BIGINT,
    rejected_count BIGINT,

    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    updated_at TIMESTAMP,

    failed_step STRING,
    error_message STRING,
    skipped_reason STRING,
    original_success_run_id STRING
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {config.quarantine_table} (
    batch_id STRING,
    entity STRING,
    ingestion_run_id STRING,
    source_file STRING,
    file_checksum STRING,
    contract_version STRING,
    raw_record STRING,
    rejection_reason STRING,
    quarantined_at TIMESTAMP
)
USING DELTA
""")

for entity, spec in contract["entities"].items():
    empty_source = spark.createDataFrame(
        [],
        contract["schemas"][entity],
    )

    projected = project_contract(empty_source, spec)

    definitions = [
        f"`{field.name}` {field.dataType.simpleString()}"
        for field in projected.schema
    ]

    definitions.extend(
        f"`{name}` {dtype}"
        for name, dtype in METADATA.items()
    )

    spark.sql(
        f"CREATE TABLE IF NOT EXISTS "
        f"{config.bronze_table(entity)} "
        f"({', '.join(definitions)}) USING DELTA"
    )

print("Setup complete:", config.namespace)
print("Contract:", contract["version"])